# C11x · Phân tích 12 run cũ trên val (không train)

**Mục đích:** biết bài có nội dung đáng viết không, **trước khi** tốn GPU cho seed 4–5.

**Vì sao dùng val:** recall theo GT trên tập test và seed 4–5 vẫn chưa bị xem. Notebook này **không chạm tập test**.

**Cách chạy:** Runtime → **L4 hoặc A100** → **Run all**. Bị ngắt thì Run all lại; phần đã xong tự bỏ qua.

**Tiêu chí quyết định, ghi trước khi xem kết quả** (`scripts/rq2/explore_decision.py`; ngưỡng là [design], không phải phép kiểm thống kê, không dùng làm kết luận trong bài). Tính trên GT rất nhỏ, trung bình theo seed:

| Tiêu chí | Đạt khi |
|---|---|
| C1 · Có cứu thật | Với GT mà A0 không cho ứng viên: recall(A2) − recall(A0) > 0 ở mọi seed, trung bình ≥ 0,02 (score ≥ 0,25) |
| C2 · Rõ lý do | Trong GT rất nhỏ mà A2 chưa phát hiện được: một loại lỗi chiếm ≥ 50% (score ≥ 0,05) |
| C3 · Giữ A8 | [A2 − A8] ở cụm dày trừ ở GT cô lập: cùng dấu ở mọi seed, độ lớn trung bình ≥ 0,01 |

**Kết quả:** đạt cả C1 và C2 → train. Đạt một → nhóm quyết. Không đạt cái nào → chưa train. C3 trượt → bỏ A8, còn 4 run thay vì 6.

**Khi xong:** gửi lại nội dung ô cuối cùng (và thư mục `c11x-explore-val` nếu được).

## 0. Cấu hình

Đường dẫn giống `C11_colab.ipynb`. Code cũ (`scripts/c11`) đọc từ **Drive**; code mới (`scripts/c11x`) lấy từ
**GitHub** (`BRANCH`). Đổi `BRANCH` thành `main` sau khi PR được merge.

In [ ]:
from pathlib import Path
import sys, json, time, subprocess

DRIVE      = Path("/content/drive/MyDrive/NCKH")
CODE_DIR   = DRIVE / "repo/NCKH/scripts/c11"          # code C11 trên Drive (không có trên GitHub)
DATA_DRIVE = DRIVE / "data"
EXP_DRIVE  = DRIVE / "repo/NCKH/experiments"
LOCAL      = Path("/content/aitodr")
PROTOCOL, VERSION = "woX", "oc"
RUNS_OLD   = EXP_DRIVE / "runs-woX"                    # 12 run thăm dò của C11 (A0/A1/A8/A2 × seed 1–3)
RUNS_NEW   = EXP_DRIVE / "runs-woX-c11x"               # run mới theo preregistration
BRANCH     = "admin/c11-novelty-plan"                  # đổi thành "main" sau khi merge
GH_DIR     = Path("/content/nckh_gh")
PERGT_DIR  = EXP_DRIVE / "c11x-pergt-val"              # bảng theo GT trên val của 12 run cũ (dùng chung 2 notebook)
GATE_FILE  = EXP_DRIVE / "c11x-h2-gate.json"           # kết quả cổng H2 (preregistration §5)

class StopRun(Exception):
    """Dừng có chủ đích, kèm lý do (Run all dừng ở ô này)."""

def today():
    """Ngày lúc ghi file (không cố định từ đầu phiên như biến TODAY của C11)."""
    return time.strftime("%Y-%m-%d")

from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "CPU runtime"

BRANCH = "admin/journal-diagnostic"                                   # đổi thành "main" sau khi merge
ARMS, SEEDS = ["A0", "A1", "A8", "A2"], [1, 2, 3]
OUT_DIR = EXP_DRIVE / "c11x-explore-val"

In [ ]:
!pip -q install "ultralytics==8.4.174" shapely gdown statsmodels
!rm -rf "{GH_DIR}" && git clone -q --depth 1 -b "{BRANCH}" https://github.com/VoDucNhatku/nckh-group7-tiny-obb "{GH_DIR}"
GH_COMMIT = subprocess.check_output(["git", "-C", str(GH_DIR), "rev-parse", "HEAD"]).decode().strip()
!rm -rf /content/c11 && cp -r "{CODE_DIR}" /content/c11
for p in ("/content/c11", str(GH_DIR / "scripts/c11x")):
    if p not in sys.path:
        sys.path.insert(0, p)
import ultralytics, torch, numpy, platform
ENV = dict(ultralytics=ultralytics.__version__, torch=torch.__version__, numpy=numpy.__version__,
           python=platform.python_version(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           c11x_commit=GH_COMMIT)
print(ENV)
assert ENV["ultralytics"] == "8.4.174"

sys.path.insert(0, str(GH_DIR / "scripts/rq2"))
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "test_errors_did.py", "test_explore.py", "-p", "no:cacheprovider"],
                   cwd=str(GH_DIR / "scripts/rq2"), capture_output=True, text=True)
print(r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-500:])
if r.returncode != 0:
    print(r.stdout[-5000:], r.stderr[-1000:])
    raise StopRun("Test hỏng. Sửa trước khi chạy tiếp.")
missing = [f"c11_{a}_s{s}" for a in ARMS for s in SEEDS if not (RUNS_OLD / f"c11_{a}_s{s}" / "weights" / "last.pt").exists()]
if missing:
    raise StopRun(f"Thiếu checkpoint trên Drive: {missing}")
print("đủ 12 checkpoint")

## 1. Dữ liệu và dataset YOLO (giống hệt C11)

Giải nén nhãn + ảnh woX từ Drive, dựng lại `yolo_woX` bằng `aitodr_data.build_yolo_dataset` với **cùng `seed=0`**,
rồi so với `split_manifest.json` đã lưu ngày 20/10. Lệch thì dừng: mọi so sánh với 12 run cũ cần đúng split cũ.

In [ ]:
LOCAL.mkdir(parents=True, exist_ok=True)
lab_zip = DATA_DRIVE / "labels.zip"
assert lab_zip.exists() and open(lab_zip, "rb").read(2) == b"PK", "labels.zip chưa có trong DATA_DRIVE"
!unzip -q -o "{lab_zip}" -d "{LOCAL}"
LABELS = next(p.parent for p in LOCAL.rglob("trainval") if p.is_dir() and any(p.glob("*.txt")))
IMAGES = LOCAL / "images_raw"
IMAGES.mkdir(exist_ok=True)
for z in ["aitod_wo_xview_trainval_imgs.zip", "aitod_wo_xview_test_imgs.zip"]:
    src = DATA_DRIVE / z
    if src.exists() and not (IMAGES / (z + ".done")).exists():
        !cp "{src}" /content/ && unzip -q -o "/content/{z}" -d "{IMAGES}" && rm "/content/{z}" && touch "{IMAGES}/{z}.done"
    print(z, "OK" if (IMAGES / (z + ".done")).exists() else "CHƯA CÓ")

import aitodr_data as ad
XVIEW = ad.xview_stems()
EXCLUDE = (XVIEW["trainval"] | XVIEW["test"]) if PROTOCOL == "woX" else set()
DS = LOCAL / f"yolo_{PROTOCOL}"
man = ad.build_yolo_dataset(LABELS, IMAGES, DS, version=VERSION, exclude_xview=(PROTOCOL == "woX"),
                            val_frac=0.10, seed=0)
old = json.loads(next(EXP_DRIVE.glob(f"*-dataset-{PROTOCOL}/split_manifest.json")).read_text())
assert man["splits"] == old["splits"], ("split khác ngày 20/10", man["splits"], old["splits"])
DATA_YAML = DS / "aitodr.yaml"
print("split khớp:", man["splits"])

## 2. Bảng theo GT từ nhãn, chỉ giữ ảnh val

In [ ]:
import pandas as pd, rq2_errors as re_, rq2_did as did, predict_split as ps, explore_decision as ed
import c11x_results as cr
from aitodr_data import scene_key
RQ2_DIR = GH_DIR / "scripts/rq2"
OFF = OUT_DIR / "rq2_trainval"
if not (OFF / "rq2_per_gt.csv.gz").exists():
    subprocess.run([sys.executable, "rq2_offline.py", "--labels", str(LABELS / "trainval"), "--out", str(OFF),
                    "--protocol", "woX", "--arms", "A0,A1,A8,A2"], cwd=str(RQ2_DIR), check=True)
VAL_IMAGES = DS / "images" / "val"
val_stems = {p.stem for p in VAL_IMAGES.iterdir()}
off = pd.read_csv(OFF / "rq2_per_gt.csv.gz")
off = off[off.stem.isin(val_stems)].copy()
stems = sorted(off.stem.unique())
print(len(val_stems), "ảnh val;", len(stems), "ảnh có GT;", len(off), "GT; rất nhỏ:", int((off.bin == "vt").sum()))
pairs = pd.DataFrame({"a": off.scene, "b": off.stem.map(scene_key)}).drop_duplicates()
if not (len(pairs) == pairs.a.nunique() == pairs.b.nunique()):
    raise StopRun(f"scene_key lệch: {len(pairs)} cặp, {pairs.a.nunique()} và {pairs.b.nunique()} nhóm.")
print(off.scene.nunique(), "cụm cảnh, khớp aitodr_data.scene_key")
off2 = off[["stem", "gt", "scene", "bin", "nn16", "A0_cand", "cls"]].assign(starved_A0=off.A0_cand == 0,
                                                                            density=re_.density_bin(off.nn16).values)
print("GT rất nhỏ mà A0 không cho ứng viên:", int((off2[off2.bin == "vt"].starved_A0).sum()))

## 3. Predict val cho 12 run, hai head (lưu trên Drive, score ≥ 0,05)

In [ ]:
import gc, torch
PRED = OUT_DIR / "pred"
for s in SEEDS:
    for a in ARMS:
        for head in ("e2e", "nms"):
            info = ps.predict_to_dota(RUNS_OLD / f"c11_{a}_s{s}" / "weights" / "last.pt", VAL_IMAGES,
                                      PRED / f"{a}_s{s}_{head}", head, conf=0.05)
            print(a, s, head, info, flush=True)
        gc.collect(); torch.cuda.empty_cache()

## 4. Số phận từng GT, recall theo IoU, M1 và M2

Chỉ để mô tả. Bootstrap B = 2.000 theo cụm cảnh, gộp bất định seed với t 2 bậc tự do, nên khoảng rất rộng.

In [ ]:
FATES = OUT_DIR / "fates"
FATES.mkdir(parents=True, exist_ok=True)

def fates(head, score):
    frames = []
    for s in SEEDS:
        for a in ARMS:
            f = FATES / f"{head}_score{score:g}_{a}_s{s}.parquet"
            if not f.exists():
                t0 = time.time()
                re_.fates_run(LABELS / "trainval", stems, PRED / f"{a}_s{s}_{head}", score).to_parquet(f)
                print(head, score, a, s, f"{time.time() - t0:.0f}s", flush=True)
            frames.append(pd.read_parquet(f).assign(arm=a, seed=s))
    df = pd.concat(frames, ignore_index=True).merge(off2, on=["stem", "gt"], how="left", validate="many_to_one")
    assert df["scene"].notna().all(), "có GT không khớp bảng nhãn"
    return df.assign(detected=lambda x: x["det_0.5"])

TABLES = {}
for head, score in (("e2e", 0.25), ("e2e", 0.05), ("nms", 0.25)):
    TABLES[(head, score)] = df = fates(head, score)
    tag = f"{head}_score{score:g}"
    for name, by in (("arm_bin", ("arm", "bin")), ("arm_bin_starved", ("arm", "bin", "starved_A0")),
                     ("arm_bin_density", ("arm", "bin", "density"))):
        re_.fate_table(df, by=by).to_csv(OUT_DIR / f"fate_{name}_{tag}.csv")
    re_.recall_curve(df).to_csv(OUT_DIR / f"recall_iou_{tag}.csv")
    print("==", tag); display(re_.fate_table(df).round(3)); display(re_.recall_curve(df).round(3))
    vt = df[df.bin == "vt"]
    mech = {"M1_A2-A0_starved_vs_not": did.did_bootstrap(vt, "A2", "A0", group="starved_A0", g1=True, g0=False, B=2000),
            "M2_A2-A8_dense_vs_isolated": did.did_bootstrap(vt, "A2", "A8", group="density", g1=">=3", g0="isolated", B=2000)}
    for k, r in mech.items():
        r["combined"] = cr.combine_scene_seed(r["scene"], r["seed"])
        print(tag, k, "Δ =", round(r["scene"]["delta"], 4), "theo seed:", {s: round(v, 4) for s, v in r["seed"]["per_seed"].items()},
              "CI gộp:", round(r["combined"]["lo"], 4), round(r["combined"]["hi"], 4))
    (OUT_DIR / f"mech_{tag}.json").write_text(json.dumps(mech, indent=2, default=float))

## 5. Quyết định theo tiêu chí đã ghi. Gửi lại nội dung ô này.

In [ ]:
res = ed.decide(TABLES[("e2e", 0.25)], TABLES[("e2e", 0.05)])
res["env"] = ENV
res["n_gt_val"] = len(off2)
res["recall_iou_e2e_0.25"] = re_.recall_curve(TABLES[("e2e", 0.25)]).xs("mean", axis=1, level=1).round(4).reset_index().to_dict("records")
(OUT_DIR / "decision.json").write_text(json.dumps(res, indent=2, ensure_ascii=False, default=float))
print(json.dumps(res, indent=1, ensure_ascii=False, default=float))